# 14 · Select well-perturbed cells

A cell can carry a guide and show no transcriptional response to it. Keeping
those cells dilutes every downstream effect estimate.

This notebook fits, per Leiden cluster, a linear model of expression on the
knockout indicators, then runs the expectation-maximisation procedure of
Dixit et al. (2016) to ask, for each cell, how much its own perturbation
explains its expression. Cells scoring above `par_em_probability_cutoff`, plus
all control cells, are kept.

**Reads** `par_save_filename_8`.
**Writes** `par_em_selected_cells_recomputed_file`.

Slow: one model per cluster, then one leave-one-covariate-out prediction per
knockout gene per cluster.

:::{note}
This EM is slow across all knockouts and all clusters, so its result is
provided rather than regenerated: `TextFiles/selectedCellsAfterEM.csv` holds
the 242,938 cells the published analysis kept, and notebooks 15 and 17 subset
to them.

Running this notebook writes its own selection under `outputs/` and leaves the
provided list untouched.
:::


## Setup

In [ ]:
from libraries import *
from parameters import *
from pathlib import Path

os.chdir(projectDir)
from sklearn import linear_model

## The EM adjustment

For each knockout covariate, the cells carrying it are refit with the covariate
zeroed out, and the residual sum of squares is compared with and without it. A
cell whose expression is explained better with its knockout in the model scores
high; one that looks the same either way scores low.

In [ ]:
def bayes_cov_col(Y, X, cols, lm):
    """Per-cell probability that a cell's own perturbation is real."""
    Yhat = pd.DataFrame(lm.predict(X), index=Y.index, columns=Y.columns)
    SSE_all = np.square(Y.subtract(Yhat))
    X_adjust = X.copy()

    for curcov in cols:
        curcells = X[X[curcov] > 0].index
        if len(curcells) <= 2:
            continue

        X_notcur = X.copy()
        X_notcur.loc[:, curcov] = 0
        X_sub = X_notcur.loc[curcells]
        Y_sub = Y.loc[curcells]

        GENE_var = 2.0 * Y_sub.var(axis=0)
        vargenes = GENE_var[GENE_var > 0].index

        Yhat_notcur = pd.DataFrame(lm.predict(X_sub), index=Y_sub.index, columns=Y_sub.columns)
        SSE_notcur = np.square(Y_sub.subtract(Yhat_notcur))
        SSE = SSE_all.loc[curcells].subtract(SSE_notcur)

        SSE_transform = SSE.div(GENE_var + 0.5)[vargenes].sum(axis=1)
        X_adjust.loc[curcells, curcov] = np.divide(1.0, 1.0 + np.exp(SSE_transform))

    return X_adjust

## Run the EM per cluster

The EM runs on the `ClusterResiduals` layer from notebook 13. A cell carrying
no knockout guide is a control: its covariate row sums to zero, and controls
are kept unconditionally.

In [ ]:
adata = sc.read(par_save_filename_8)
print(f"input: {adata.shape[0]} cells x {adata.shape[1]} genes")

clusters = sorted(adata.obs["leiden"].unique(), key=int)
covariates = [
    c for c in adata.uns["feature_barcode_names_filtered_GENES"] if c != "GENE_CONTROL_"
]

selected, counts = [], []

for cluster in clusters:
    sub = adata[adata.obs["leiden"] == cluster, :]
    X = sub.obs[covariates].astype(float)
    Y = pd.DataFrame(sub.layers["ClusterResiduals"], index=sub.obs_names,
                     columns=sub.var_names)

    # Most knockouts are absent from any one cluster. Their columns are all
    # zero, which makes the design rank deficient and the least-squares solve
    # fail to converge. They carry no information here either way, so the fit
    # is restricted to the knockouts this cluster actually contains.
    present = [c for c in X.columns if X[c].sum() > 0]
    X = X[present]

    lm = linear_model.LinearRegression().fit(np.array(X), np.array(Y))
    X_adjust = bayes_cov_col(Y, X, present, lm)

    control_cells = X.index[X.sum(axis=1) == 0]
    probs = X_adjust.sum(axis=1)
    kept = list(dict.fromkeys(list(control_cells) +
                              list(probs[probs > par_em_probability_cutoff].index)))

    selected.extend(kept)
    counts.append({"cluster": cluster, "cells": sub.shape[0],
                   "controls": len(control_cells), "kept": len(kept)})
    print(f"  cluster {cluster}: {sub.shape[0]} cells, "
          f"{len(present)} knockouts present -> {len(kept)} kept")

print()
print(pd.DataFrame(counts).to_string(index=False))

## Subset and write

In [ ]:
selected = pd.Index(dict.fromkeys(selected))
print(f"cells kept: {len(selected)} of {adata.shape[0]} "
      f"({100 * len(selected) / adata.shape[0]:.1f}%)")

# The selection itself is the result of this step. The object it would produce
# is simply notebook 13's output subset to these cells, so notebooks 15 and 17
# each subset it themselves rather than a large intermediate being written.
Path(par_em_selected_cells_recomputed_file).parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame({"x": list(selected)}).to_csv(par_em_selected_cells_recomputed_file,
                                           index=False)
print(f"written: {par_em_selected_cells_recomputed_file}")
print(f"the list used downstream is {par_em_selected_cells_file}, left unchanged")